In [9]:
import json

# Assuming your JSON is saved as 'samples.json'
with open('new_definitions.json', 'r') as f:
    data = json.load(f)

# Define the UL campaign replacements to keep the table clean
campaign_map = {
    "RunIISummer20UL16NanoAODAPVv9-": "[16APV]",
    "RunIISummer20UL16NanoAODv9-": "[16]",
    "RunIISummer20UL17NanoAODv9-": "[17]",
    "RunIISummer20UL18NanoAODv9-": "[18]"
}

# Standardize year labels for the table column
year_labels = {
    "2016_PreVFP": "2016 APV",
    "2016_PostVFP": "2016",
    "2017": "2017",
    "2018": "2018"
}

def tex_escape(text):
    # Escapes underscores and adds \allowbreak{} so LaTeX can wrap long strings
    # The {} prevents LaTeX from absorbing the next letter into the macro name.
    text = text.replace('_', '\\_\\allowbreak{}')
    text = text.replace('-', '-\\allowbreak{}')
    return text

def chunk_list(lst, n):
    """Yield successive n-sized chunks from a list."""
    for i in range(0, len(lst), n):
        yield lst[i:i + n]

def generate_latex_table(sample_keys, title, label):
    print("\\begin{table}[hbtp]")
    print("  \\scriptsize")
    print("  \\centering")
    print(f"  \\topcaption{{{title}}}")
    print(f"  \\label{{{label}}}")
    print("  \\renewcommand{\\arraystretch}{1.4}")
    print("  \\begin{tabular}{lcp{0.65\\textwidth}}")
    print("    \\hline\\hline")
    print("    Process & Year & Dataset \\\\")
    print("    \\hline\\hline")

    for key in sample_keys:
        info = data[key]
        display_name = tex_escape(key)
        files = info.get("files", [])
        
        # Sort files chronologically: 16APV -> 16 -> 17 -> 18
        files.sort(key=lambda x: (x["metadata"]["year"].replace("PreVFP", "0"), x["metadata"]["year"]))

        for i, f in enumerate(files):
            year_raw = f["metadata"]["year"]
            year_str = year_labels.get(year_raw, year_raw)

            das_name = f["das_names"][0]
            parts = das_name.split('/')
            primary_dataset = tex_escape(parts[1])
            campaign_gt = parts[2]

            for camp, token in campaign_map.items():
                if campaign_gt.startswith(camp):
                    campaign_gt = campaign_gt.replace(camp, token)
                    break

            campaign_gt = tex_escape(campaign_gt)
            
            process_col = display_name if i == 0 else ""

            # Drops the datatier (e.g., NANOAODSIM) to save horizontal space
            print(f"    {process_col:<25} & {year_str:<8} & \\texttt{{/{primary_dataset}/\\newline {campaign_gt}}} \\\\")

        if files:
            xsec = files[0]["metadata"]["xsec"]
            print(f"    \\multicolumn{{3}}{{l}}{{\({xsec}\\,\\mathrm{{pb}}\)}} \\\\[3pt]")
            print("    \\hline")

    print("    \\multicolumn{3}{l}{Campaign:} \\\\")
    print("      \\multicolumn{3}{l}{[16APV] = RunIISummer20UL16NanoAODAPVv9-} \\\\")
    print("      \\multicolumn{3}{l}{[16] = RunIISummer20UL16NanoAODv9-} \\\\")
    print("      \\multicolumn{3}{l}{[17] = RunIISummer20UL17NanoAODv9-} \\\\")
    print("      \\multicolumn{3}{l}{[18] = RunIISummer20UL18NanoAODv9-} \\\\")
    print("    \\hline\\hline")
    print("  \\end{tabular}")
    print("  \\renewcommand{\\arraystretch}{1.0}")
    print("\\end{table}\n")

table_groups = [
    {
        "keys": ["ttHTobb", "ttHToNonbb"],
        "title": "\(\\ttH\) signal samples and cross sections.",
        "label": "tab:samples:signal:ttH"
    },
    {
        "keys": ["TTToHadronic", "TTToSemiLeptonic", "TTTo2L2Nu", "TTbb_Hadronic", "TTbb_SemiLeptonic", "TTbb_2L2Nu"],
        "title": "Nominal \(\\ttbar\) and \(\\ttbar+b\\bar{b}\) background samples and cross sections.",
        "label": "tab:samples:bkg:ttbar"
    },
    {
        "keys": ["TTZToQQ", "TTZToBB", "TTZToLLNuNu", "TTWJetsToLNu", "TTWJetsToQQ", "TTTT", "TTGJets"],
        "title": "Associated top quark production (\(\\ttbar+\\text{X}\)) background samples and cross sections.",
        "label": "tab:samples:bkg:ttX"
    },
    {
        "keys": ["ST_tW_top", "ST_tW_antitop", "ST_s_channel", "ST_t_channel_top", "ST_t_channel_antitop"],
        "title": "Single top quark production background samples and cross sections.",
        "label": "tab:samples:bkg:singletop"
    },
    {
        "keys": [k for k in data.keys() if k.startswith("QCD")],
        "title": "QCD multi-jet background samples and cross sections.",
        "label": "tab:samples:bkg:qcd"
    },
    {
        "keys": [k for k in data.keys() if k.startswith("WJets")],
        "title": "W+Jets background samples and cross sections, binned by \(H_T\).",
        "label": "tab:samples:bkg:wjets"
    },
    {
        "keys": [k for k in data.keys() if k.startswith("DYJets")],
        "title": "Drell-Yan (DY+Jets) background samples and cross sections, binned by \(H_T\).",
        "label": "tab:samples:bkg:dyjets"
    }
]

# Set maximum number of samples (processes) per table
MAX_SAMPLES_PER_TABLE = 4

for group in table_groups:
    valid_keys = [k for k in group["keys"] if k in data]
    if not valid_keys:
        continue
        
    # Split the valid keys into chunks
    chunks = list(chunk_list(valid_keys, MAX_SAMPLES_PER_TABLE))
    
    for i, chunk in enumerate(chunks):
        title = group["title"]
        label = group["label"]
        
        # If there are multiple chunks, append part numbers to titles and labels
        if len(chunks) > 1:
            title = title.rstrip('.') + f" (Part {i+1})."
            label = f"{label}_{i+1}"
            
        generate_latex_table(chunk, title, label)

\begin{table}[hbtp]
  \scriptsize
  \centering
  \topcaption{\(\ttH\) signal samples and cross sections.}
  \label{tab:samples:signal:ttH}
  \renewcommand{\arraystretch}{1.4}
  \begin{tabular}{lcp{0.65\textwidth}}
    \hline\hline
    Process & Year & Dataset \\
    \hline\hline
    ttHTobb                   & 2016 APV & \texttt{/ttHTobb\_\allowbreak{}M125\_\allowbreak{}TuneCP5\_\allowbreak{}13TeV-\allowbreak{}powheg-\allowbreak{}pythia8/\newline [16APV]106X\_\allowbreak{}mcRun2\_\allowbreak{}asymptotic\_\allowbreak{}preVFP\_\allowbreak{}v11-\allowbreak{}v2} \\
                              & 2016     & \texttt{/ttHTobb\_\allowbreak{}M125\_\allowbreak{}TuneCP5\_\allowbreak{}13TeV-\allowbreak{}powheg-\allowbreak{}pythia8/\newline [16]106X\_\allowbreak{}mcRun2\_\allowbreak{}asymptotic\_\allowbreak{}v17-\allowbreak{}v2} \\
                              & 2017     & \texttt{/ttHTobb\_\allowbreak{}M125\_\allowbreak{}TuneCP5\_\allowbreak{}13TeV-\allowbreak{}powheg-\allowbreak{}pythia8/\newli